# BERT + QLoRA Fine-tuning for Keyword Extraction

Fine-tunes `google-bert/bert-base-uncased` with QLoRA (4-bit + LoRA via PEFT)  
for token classification (BIO tagging) to extract keywords from product descriptions.

**Task**: Given a product description, tag each token as:
- `B-KW` — beginning of a keyword phrase
- `I-KW` — inside a keyword phrase  
- `O`    — not a keyword

### 1. Install dependencies

In [ ]:
# !pip install transformers datasets peft bitsandbytes accelerate seqeval -q

### 2. Imports

In [ ]:
import ast
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from peft import LoraConfig, get_peft_model, TaskType
from seqeval.metrics import classification_report, f1_score
from transformers.utils.notebook import NotebookProgressCallback

MODEL_PATH = "google-bert/bert-base-uncased"
CSV_PATH   = "/Users/rishirajsaikia/Downloads/bert_train/item_description_extracted_keys.csv"
OUTPUT_DIR = "./bert-qlora-keyword"

LABEL_LIST = ["O", "B-KW", "I-KW"]
label2id   = {l: i for i, l in enumerate(LABEL_LIST)}
id2label   = {i: l for i, l in enumerate(LABEL_LIST)}

### 3. Load & preview data

In [9]:
# CSV has header row: 'Item Description', 'Extracted Key'
df = pd.read_csv(
    CSV_PATH
)
df = df.dropna().reset_index(drop=True)
print(f"Total samples: {len(df)}")
df.head(3)

Total samples: 325618


,Item Description,Extracted Key
0,Bajaj PX 97 Torque New 36L Personal Air Cooler...,personal air cooler
1,Aquaguard Delight NXT Aquasaver 9-Stage Water ...,water purifier
2,Philips PowerPro FC9352/01 Compact Bagless Vac...,vacuum cleaner


### 4. Convert to BIO token labels

Strategy: tokenize description with whitespace, find the keyword span, assign BIO tags.

In [10]:
def bio_tag(description: str, keyword: str):
    """Assign BIO tags to whitespace-tokenized description words."""
    words  = description.lower().split()
    kw_words = keyword.lower().split()
    n_kw = len(kw_words)
    tags = ["O"] * len(words)

    for i in range(len(words) - n_kw + 1):
        if words[i:i + n_kw] == kw_words:
            tags[i] = "B-KW"
            for j in range(1, n_kw):
                tags[i + j] = "I-KW"
            break  # tag first occurrence only

    return words, tags

# Quick sanity check
desc = "Aquaguard Delight NXT Aquasaver 9-Stage Water Purifier | Upto 60% Water Savings"
kw   = "water purifier"
w, t = bio_tag(desc, kw)
list(zip(w, t))

[('aquaguard', 'O'),
 ('delight', 'O'),
 ('nxt', 'O'),
 ('aquasaver', 'O'),
 ('9-stage', 'O'),
 ('water', 'B-KW'),
 ('purifier', 'I-KW'),
 ('|', 'O'),
 ('upto', 'O'),
 ('60%', 'O'),
 ('water', 'O'),
 ('savings', 'O')]

In [12]:
records = []
for _, row in df.iterrows():
    words, tags = bio_tag(str(row["Item Description"]), str(row["Extracted Key"]))
    records.append({"words": words, "tags": tags})

raw_dataset = Dataset.from_list(records)

# Train / val / test split  80 / 10 / 10
splits   = raw_dataset.train_test_split(test_size=0.1, seed=42)
val_test = splits["test"].train_test_split(test_size=0.5, seed=42)
dataset  = DatasetDict({
    "train":      splits["train"],
    "validation": val_test["train"],
    "test":       val_test["test"],
})
dataset

DatasetDict({
    train: Dataset({
        features: ['words', 'tags'],
        num_rows: 293056
    })
    validation: Dataset({
        features: ['words', 'tags'],
        num_rows: 16281
    })
    test: Dataset({
        features: ['words', 'tags'],
        num_rows: 16281
    })
})

### 5. Tokenize & align labels

BERT's WordPiece splits words into sub-tokens. We propagate the label of the first sub-token and set `-100` for the rest (ignored in loss).

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["words"],
        truncation=True,
        max_length=128,
        is_split_into_words=True,
    )
    all_labels = []
    for i, tag_seq in enumerate(examples["tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        labels, prev_word_id = [], None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)          # [CLS] / [SEP]
            elif wid != prev_word_id:
                labels.append(label2id[tag_seq[wid]])  # first sub-token
            else:
                labels.append(-100)          # subsequent sub-tokens
            prev_word_id = wid
        all_labels.append(labels)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_and_align,
    batched=True,
    remove_columns=["words", "tags"],
)
tokenized_dataset

Map: 100%|██████████| 16281/16281 [00:00<00:00, 20175.30 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 293056
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 16281
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 16281
    })
})

### 6. Load model with LoRA (Mac/MPS compatible)

In [14]:
# Mac/MPS: no bitsandbytes — use plain LoRA with MPS device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

base_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_PATH,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
).to(device)
base_model.config.use_cache = False
print(base_model)

Using device: mps


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9089.67it/s]
BertForTokenClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not o

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

### 7. Apply LoRA adapters

In [15]:
lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=16,                          # rank
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    # target the attention projection matrices
    target_modules=["query", "key", "value", "dense"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 2,656,515 || all params: 111,550,470 || trainable%: 2.3814


### 8. Metrics

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels, true_preds = [], []
    for pred_row, label_row in zip(predictions, labels):
        true_labels.append([id2label[l] for l in label_row if l != -100])
        true_preds.append([id2label[p] for p, l in zip(pred_row, label_row) if l != -100])

    return {
        "f1": f1_score(true_labels, true_preds),
        "report": classification_report(true_labels, true_preds),
    }

### 9. Training

In [19]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=False,
    logging_steps=100,
    report_to="none",
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class =tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/opt/anaconda3/envs/p3.12/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1,Report
1,0.011589,0.009366,0.973194,precision recall f1-score support KW 0.96 0.98 0.97 10403 micro avg 0.96 0.98 0.97 10403 macro avg 0.96 0.98 0.97 10403 weighted avg 0.96 0.98 0.97 10403
2,0.009486,0.005710,0.984259,precision recall f1-score support KW 0.98 0.99 0.98 10403 micro avg 0.98 0.99 0.98 10403 macro avg 0.98 0.99 0.98 10403 weighted avg 0.98 0.99 0.98 10403
3,0.005188,0.004221,0.989766,precision recall f1-score support KW 0.98 0.99 0.99 10403 micro avg 0.98 0.99 0.99 10403 macro avg 0.98 0.99 0.99 10403 weighted avg 0.98 0.99 0.99 10403


/opt/anaconda3/envs/p3.12/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/envs/p3.12/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=54948, training_loss=0.019703969075736707, metrics={'train_runtime': 12443.968, 'train_samples_per_second': 70.65, 'train_steps_per_second': 4.416, 'total_flos': 2.347395155558669e+16, 'train_loss': 0.019703969075736707, 'epoch': 3.0})

### 10. Evaluate on test set

In [24]:
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

results = trainer.evaluate(eval_dataset=tokenized_dataset["test"], metric_key_prefix="test")
print(results)

{'test_loss': 0.004035189747810364, 'test_f1': 0.9889607547536838, 'test_report': '              precision    recall  f1-score   support\n\n          KW       0.98      0.99      0.99     10232\n\n   micro avg       0.98      0.99      0.99     10232\n   macro avg       0.98      0.99      0.99     10232\nweighted avg       0.98      0.99      0.99     10232\n', 'test_runtime': 89.4468, 'test_samples_per_second': 182.019, 'test_steps_per_second': 5.691, 'epoch': 3.0}


### 11. Save model

In [23]:
model.save_pretrained("./bert-lora-keyword")
tokenizer.save_pretrained("./bert-lora-keyword")
print(f"Model saved to ./bert-lora-keyword")

Model saved to ./bert-lora-keyword


In [22]:
#push2hub
trainer.push_to_hub()

Processing Files (2 / 2): 100%|██████████| 10.7MB / 10.7MB, 1.52MB/s  
New Data Upload: 100%|██████████| 10.7MB / 10.7MB, 1.52MB/s  


CommitInfo(commit_url='https://huggingface.co/rishi19/bert-qlora-keyword/commit/97cc1b1987dc0678284499b34b76103e6addbad5', commit_message='End of training', commit_description='', oid='97cc1b1987dc0678284499b34b76103e6addbad5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rishi19/bert-qlora-keyword', endpoint='https://huggingface.co', repo_type='model', repo_id='rishi19/bert-qlora-keyword'), pr_revision=None, pr_num=None)

### 12. Inference — extract keywords from a new description

In [26]:
from peft import PeftModel

def load_model(output_dir=OUTPUT_DIR):
    tok = AutoTokenizer.from_pretrained(output_dir)
    base = AutoModelForTokenClassification.from_pretrained(
        MODEL_PATH,
        num_labels=len(LABEL_LIST),
        id2label=id2label,
        label2id=label2id,
    ).to(device)
    m = PeftModel.from_pretrained(base, output_dir)
    m.eval()
    return tok, m

def extract_keywords(description: str, tok, m) -> list[str]:
    words = description.lower().split()
    inputs = tok(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(device)

    with torch.no_grad():
        logits = m(**inputs).logits

    pred_ids  = logits.argmax(-1)[0].tolist()
    word_ids  = inputs.word_ids(batch_index=0)

    # Map predictions back to words (first sub-token only)
    word_preds, seen = {}, set()
    for idx, wid in enumerate(word_ids):
        if wid is not None and wid not in seen:
            word_preds[wid] = id2label[pred_ids[idx]]
            seen.add(wid)

    # Reconstruct keyword spans from BIO tags
    keywords, current = [], []
    for wid in sorted(word_preds):
        tag = word_preds[wid]
        if tag == "B-KW":
            if current:
                keywords.append(" ".join(current))
            current = [words[wid]]
        elif tag == "I-KW" and current:
            current.append(words[wid])
        else:
            if current:
                keywords.append(" ".join(current))
            current = []
    if current:
        keywords.append(" ".join(current))

    return keywords


# --- Test it ---
tok, m = load_model()

test_desc = "Bajaj gheto fin service oklahoma 9-Stage Water Purifier | Upto 60% Water Savings | RO+UV+UF+MC Tech | Taste Adjuster | Suitable for Borewell, Tanker & Municipal Water | India's #1 Water Purifier"
kws = extract_keywords(test_desc, tok, m)
print("Extracted keywords:", kws)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 15896.99it/s]
BertForTokenClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 

Extracted keywords: ['water purifier']
